In [34]:
import os

import numpy as np
import pandas as pd
import torch

from google.colab import userdata
import kagglehub

In [35]:

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_API_TOKEN')

In [36]:
# Download latest version
path = kagglehub.dataset_download("brendan45774/test-file")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'test-file' dataset.
Path to dataset files: /kaggle/input/test-file


In [37]:


# List files in the downloaded dataset directory
files = os.listdir(path)
print("Files in dataset directory:", files)
print(path)

Files in dataset directory: ['tested.csv']
/kaggle/input/test-file


In [38]:
# Assuming there is a CSV file in the path
csv_path = None
csv_files = [f for f in files if f.endswith('.csv')]
if csv_files:
    csv_path = os.path.join(path, csv_files[0])
else:
    print("No CSV file found in the directory.")

In [39]:
# Read CSV using pandas
df = pd.read_csv(csv_path)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [40]:
df['Cabin'].unique()
df[df['Cabin'] == 'B52 B54 B56'] # looking at data anomalies, here there are theree cabins for the same passenger

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
372,1264,0,1,"Ismay, Mr. Joseph Bruce",male,49.0,0,0,112058,0.0,B52 B54 B56,S


In [41]:
trans_df = pd.DataFrame()

trans_df['passenger_id'] = df['PassengerId']
trans_df['survived'] = df['Survived'] # dependent variable


# cleaning passenger class
trans_df['pclass_1'] = (df['Pclass'] == 1).astype(int)
trans_df['pclass_2'] = (df['Pclass'] == 2).astype(int)
trans_df['pclass_3'] = (df['Pclass'] == 3).astype(int)

trans_df['is_male'] = (df['Sex'] == 'male').astype(int)

# scaling and cleaning the age
trans_df['age'] = df['Age']/df['Age'].max()
trans_df['age'] = trans_df['age'].fillna(trans_df['age'].mean())
trans_df['is_age_missing'] = df['Age'].isna().astype(int)

# scaling and cleaning the fare
trans_df['fare'] = df['Fare']/df['Fare'].max()
trans_df['fare'] = trans_df['fare'].fillna(trans_df['fare'].mean())
trans_df['is_fare_missing'] = df['Fare'].isna().astype(int)

# cabin data processing
decks = df['Cabin'].str[0].fillna('M')
#print(decks.unique())

trans_df['deck_a'] = (decks == 'A').astype(int)
trans_df['deck_b'] = (decks == 'B').astype(int)
trans_df['deck_c'] = (decks == 'C').astype(int)
trans_df['deck_d'] = (decks == 'D').astype(int)
trans_df['deck_e'] = (decks == 'E').astype(int)
trans_df['deck_f'] = (decks == 'F').astype(int)
trans_df['deck_g'] = (decks == 'G').astype(int)
trans_df['deck_m'] = (decks == 'M').astype(int)

# embarked data processing



trans_df['embarked_at_q'] = (df['Embarked'] == 'Q').astype(int)
trans_df['embarked_at_s'] = (df['Embarked'] == 'S').astype(int)
trans_df['embarked_at_c'] = (df['Embarked'] == 'C').astype(int)
trans_df['bias'] = 1



In [42]:
trans_df.head()

,passenger_id,survived,pclass_1,pclass_2,pclass_3,is_male,age,is_age_missing,fare,is_fare_missing,...,deck_c,deck_d,deck_e,deck_f,deck_g,deck_m,embarked_at_q,embarked_at_s,embarked_at_c,bias
0,892,0,0,0,1,1,0.453947,0,0.015282,0,...,0,0,0,0,0,1,1,0,0,1
1,893,1,0,0,1,0,0.618421,0,0.013663,0,...,0,0,0,0,0,1,0,1,0,1
2,894,0,0,1,0,1,0.815789,0,0.018909,0,...,0,0,0,0,0,1,1,0,0,1
3,895,0,0,0,1,1,0.355263,0,0.016908,0,...,0,0,0,0,0,1,0,1,0,1
4,896,1,0,0,1,0,0.289474,0,0.023984,0,...,0,0,0,0,0,1,0,1,0,1


In [47]:
# Convert inputs to float32 tensors
X_tensor = torch.tensor(trans_df.drop(columns=['passenger_id', 'survived', 'bias']).values, dtype=torch.float32)
y_tensor = torch.tensor(trans_df['survived'].values, dtype=torch.float32).unsqueeze(1)

# Set random seed for reproducibility
torch.manual_seed(42)

n_samples, n_features = X_tensor.shape
hidden_dim = 16

# 1. Initialize weights with proper scaling as leaf tensors (setting requires_grad after scaling)
W1 = torch.randn(n_features, hidden_dim) * (2 / n_features) ** 0.5
W1.requires_grad_(True)
b1 = torch.zeros(hidden_dim, requires_grad=True)

W2 = torch.randn(hidden_dim, 1) * (2 / hidden_dim) ** 0.5
W2.requires_grad_(True)
b2 = torch.zeros(1, requires_grad=True)

learning_rate = 0.05
epochs = 300

# 2. Training Loop
for epoch in range(epochs):
    # Forward Pass
    z1 = torch.matmul(X_tensor, W1) + b1
    a1 = torch.relu(z1)                     # Activation
    z2 = torch.matmul(a1, W2) + b2
    y_pred = torch.sigmoid(z2)             # Output squeezed into (0, 1)

    # Binary Cross-Entropy Loss
    loss = -torch.mean(y_tensor * torch.log(y_pred + 1e-8) + (1 - y_tensor) * torch.log(1 - y_pred + 1e-8))

    # Backward Pass (compute gradients dLoss/dW and dLoss/db)
    loss.backward()

    # Weight Update via Gradient Descent (prevent tracking history during update)
    with torch.no_grad():
        W1 -= learning_rate * W1.grad
        b1 -= learning_rate * b1.grad
        W2 -= learning_rate * W2.grad
        b2 -= learning_rate * b2.grad

        # Zero gradients for the next iteration
        W1.grad.zero_()
        b1.grad.zero_()
        W2.grad.zero_()
        b2.grad.zero_()

    if (epoch + 1) % 50 == 0:
        predictions = (y_pred >= 0.5).float()
        acc = (predictions == y_tensor).float().mean()
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Accuracy: {acc.item():.2%}")

Epoch  50 | Loss: 0.5090 | Accuracy: 68.42%
Epoch 100 | Loss: 0.3723 | Accuracy: 82.78%
Epoch 150 | Loss: 0.2591 | Accuracy: 97.13%
Epoch 200 | Loss: 0.1775 | Accuracy: 100.00%
Epoch 250 | Loss: 0.1234 | Accuracy: 100.00%
Epoch 300 | Loss: 0.0886 | Accuracy: 100.00%
